# TechMind AI — Extracción y Perfilado de Datasets

Este notebook centraliza la **extracción desde Kaggle** de los datasets candidatos para entrenar el modelo de clasificación de contenido técnico de **TechMind AI** (Etapa 2 — Construcción del Dataset).

Para cada dataset se realiza:
1. Descarga vía `kagglehub` (adapter de Pandas).
2. Perfilado rápido: dimensiones, columnas, tipos, nulos y muestra de registros.
3. Notas de relevancia para el caso de uso (clasificación temática de texto técnico).

Al final hay una **tabla comparativa** para apoyar la decisión de qué dataset(s) usar.

> Requisito: tener configurado el archivo `kaggle.json` (credenciales de la API de Kaggle) o haber ejecutado `kagglehub.login()` previamente.

## 0. Setup

In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets] pandas
import os
import time
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

### 0.1 Descubrir los archivos reales de cada dataset

`kagglehub` necesita un `file_path` **no vacío** apuntando a un archivo concreto dentro del dataset (extensiones soportadas: `.csv`, `.tsv`, `.json`, `.parquet`, `.xlsx`, etc.). Como estos datasets traen varios archivos, primero los descargamos y listamos para saber qué nombre poner en cada `file_path` de las secciones siguientes.

La API de Kaggle a veces responde con errores transitorios (ej. `502 Bad Gateway`); por eso cada descarga reintenta unas pocas veces antes de fallar.

In [4]:
handles = [
    "stackoverflow/stacksample",
    "fabiochiusano/medium-articles",
    "hsankesara/medium-articles",
    "kutayahin/stackoverflow-programming-questions-2020-2025",
    "sunilthite/text-document-classification-dataset",
]


def descargar_con_reintentos(handle, intentos=3, espera_seg=5):
    for intento in range(1, intentos + 1):
        try:
            return kagglehub.dataset_download(handle)
        except Exception as e:
            if intento == intentos:
                raise
            print(f"   (intento {intento} falló: {e}. Reintentando en {espera_seg}s...)")
            time.sleep(espera_seg)


# Guardamos la ruta local de cada dataset (ya descargado/extraído aquí) para
# leer los CSV directamente con pandas más abajo, sin volver a descargarlos
# con kagglehub.dataset_load (eso duplicaba la descarga y causaba
# DataCorruptionError por un resume corrupto en archivos grandes).
archivos_por_dataset = {}
rutas_locales = {}
for handle in handles:
    ruta_local = descargar_con_reintentos(handle)
    archivos = sorted(os.listdir(ruta_local))
    archivos_por_dataset[handle] = archivos
    rutas_locales[handle] = ruta_local
    print(f"{handle}: ({ruta_local})")
    for archivo in archivos:
        print(f"   - {archivo}")
    print()

stackoverflow/stacksample: (C:\Users\will8\.cache\kagglehub\datasets\stackoverflow\stacksample\versions\2)
   - Answers.csv
   - Tags.csv

fabiochiusano/medium-articles: (C:\Users\will8\.cache\kagglehub\datasets\fabiochiusano\medium-articles\versions\2)
   - medium_articles.csv

hsankesara/medium-articles: (C:\Users\will8\.cache\kagglehub\datasets\hsankesara\medium-articles\versions\1)
   - articles.csv

kutayahin/stackoverflow-programming-questions-2020-2025: (C:\Users\will8\.cache\kagglehub\datasets\kutayahin\stackoverflow-programming-questions-2020-2025\versions\1)
   - stackoverflow_combined.csv
   - stackoverflow_combined_info.json
   - stackoverflow_combined_statistics.json

sunilthite/text-document-classification-dataset: (C:\Users\will8\.cache\kagglehub\datasets\sunilthite\text-document-classification-dataset\versions\1)
   - df_file.csv



### 0.2 Reparar caché corrupto (opcional)

Si una celda de carga falla con `FileNotFoundError` o `DataCorruptionError` sobre un archivo que la celda 0.1 sí listó, el caché local de `kagglehub` quedó incompleto (por ejemplo, por una descarga interrumpida). Usa `reparar_cache(handle)` para forzar una re-descarga completa de ese dataset puntual.

No la dejes corriendo por defecto: `force_download=True` vuelve a bajar el dataset completo (puede pesar GBs) cada vez que se ejecuta. Descomenta solo la línea del dataset que necesites reparar.

In [3]:
def reparar_cache(handle):
    """Fuerza una redescarga completa del dataset (bypassa un caché local corrupto)."""
    ruta_local = kagglehub.dataset_download(handle, force_download=True)
    rutas_locales[handle] = ruta_local
    archivos = sorted(os.listdir(ruta_local))
    archivos_por_dataset[handle] = archivos
    print(f"{handle} reparado: ({ruta_local})")
    for archivo in archivos:
        print(f"   - {archivo}")
    return ruta_local


# Descomenta la línea del dataset que necesites reparar, por ejemplo:
# reparar_cache("stackoverflow/stacksample")

In [6]:
def perfilar(nombre, df):
    """Resumen rápido de un dataset recién cargado."""
    print(f"=== {nombre} ===")
    print("Shape:", df.shape)
    print("\nColumnas y tipos:")
    print(df.dtypes)
    print("\nNulos por columna:")
    print(df.isna().sum())
    print("\nPrimeros 5 registros:")
    display(df.head())
    return {
        "dataset": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "nombres_columnas": list(df.columns),
    }

resumenes = []

---
## 1. StackOverflow StackSample
`stackoverflow/stacksample`

Preguntas y respuestas de Stack Overflow con etiquetas (tags). Útil como fuente de contenido técnico ya categorizado por tema/tecnología.

Confirmado (ver 0.1): contiene `Questions.csv`, `Answers.csv` y `Tags.csv`. Se carga `Questions.csv` (título + cuerpo de la pregunta); `Tags.csv` se puede unir después por `Id` para obtener las categorías.

In [8]:
# Set the path to the file you'd like to load
file_path = "Questions.csv"

# Leemos directo del archivo ya descargado por la celda 0.1 (evita doble
# descarga y el DataCorruptionError de kagglehub.dataset_load en este archivo grande)
df_stacksample = pd.read_csv(
    os.path.join(rutas_locales["stackoverflow/stacksample"], file_path),
    encoding="latin1",
)

resumenes.append(perfilar("stackoverflow/stacksample", df_stacksample))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\will8\\.cache\\kagglehub\\datasets\\stackoverflow\\stacksample\\versions\\2\\Questions.csv'

---
## 2. Medium Articles (fabiochiusano)
`fabiochiusano/medium-articles`

Artículos de Medium con título, texto y tags. Buena fuente de artículos técnicos largos con temática variada.

In [7]:
# Set the path to the file you'd like to load
file_path = "medium_articles.csv"

# Leemos directo del archivo ya descargado por la celda 0.1
df_medium_chiusano = pd.read_csv(
    os.path.join(rutas_locales["fabiochiusano/medium-articles"], file_path)
)

resumenes.append(perfilar("fabiochiusano/medium-articles", df_medium_chiusano))

=== fabiochiusano/medium-articles ===
Shape: (192368, 6)

Columnas y tipos:
title        object
text         object
url          object
authors      object
timestamp    object
tags         object
dtype: object

Nulos por columna:
title        5
text         0
url          0
authors      0
timestamp    2
tags         0
dtype: int64

Primeros 5 registros:


,title,text,url,authors,timestamp,tags
0,Mental Note Vol. 24,"Photo by Josh Riemer on Unsplash\n\nMerry Christmas and Happy Holidays, everyone!\n\nWe just wanted everyone to know...",https://medium.com/invisible-illness/mental-note-vol-24-969b6a42443f,['Ryan Fan'],2020-12-26 03:38:10.479000+00:00,"['Mental Health', 'Health', 'Psychology', 'Science', 'Neuroscience']"
1,Your Brain On Coronavirus,Your Brain On Coronavirus\n\nA guide to the curious and troubling impact of the pandemic and isolation\n\nPhoto by c...,https://medium.com/age-of-awareness/how-the-pandemic-affects-our-brain-and-mental-health-ae2ec0a9fc1d,['Simon Spichak'],2020-09-23 22:10:17.126000+00:00,"['Mental Health', 'Coronavirus', 'Science', 'Psychology', 'Neuroscience']"
2,Mind Your Nose,Mind Your Nose\n\nHow smell training can change your brain in six weeks — and why it matters.\n\nBy Ann-Sophie Barwi...,https://medium.com/neodotlife/mind-your-nose-f0b097d533bb,[],2020-10-10 20:17:37.132000+00:00,"['Biotechnology', 'Neuroscience', 'Brain', 'Wellness', 'Science']"
3,The 4 Purposes of Dreams,Passionate about the synergy between science and technology to provide better care. Check out my newsletter: science...,https://medium.com/science-for-real/the-4-purposes-of-dreams-fc6719090e75,['Eshan Samaranayake'],2020-12-21 16:05:19.524000+00:00,"['Health', 'Neuroscience', 'Mental Health', 'Psychology', 'Science']"
4,Surviving a Rod Through the Head,"You’ve heard of him, haven’t you? Phineas Gage. The railroad worker who survived an explosion that involved an iron ...",https://medium.com/live-your-life-on-purpose/surviving-a-rod-through-the-head-2e5d74db978,['Rishav Sinha'],2020-02-26 00:01:01.576000+00:00,"['Brain', 'Health', 'Development', 'Psychology', 'Science']"


---
## 3. Medium Articles (hsankesara)
`hsankesara/medium-articles`

Dataset alternativo de artículos de Medium, de menor tamaño. Sirve como comparación o complemento del dataset anterior.

In [8]:
# Set the path to the file you'd like to load
file_path = "articles.csv"

# Leemos directo del archivo ya descargado por la celda 0.1
df_medium_hsankesara = pd.read_csv(
    os.path.join(rutas_locales["hsankesara/medium-articles"], file_path)
)

resumenes.append(perfilar("hsankesara/medium-articles", df_medium_hsankesara))

=== hsankesara/medium-articles ===
Shape: (337, 6)

Columnas y tipos:
author          object
claps           object
reading_time     int64
link            object
title           object
text            object
dtype: object

Nulos por columna:
author          0
claps           0
reading_time    0
link            0
title           0
text            0
dtype: int64

Primeros 5 registros:


,author,claps,reading_time,link,title,text
0,Justin Lee,8.3K,11,https://medium.com/swlh/chatbots-were-the-next-big-thing-what-happened-5fc49dd6fa61?source=---------0----------------,Chatbots were the next big thing: what happened? – The Startup – Medium,"Oh, how the headlines blared:\nChatbots were The Next Big Thing.\nOur hopes were sky high. Bright-eyed and bushy-tai..."
1,Conor Dewey,1.4K,7,https://towardsdatascience.com/python-for-data-science-8-concepts-you-may-have-forgotten-i-did-825966908393?source=-...,Python for Data Science: 8 Concepts You May Have Forgotten,"If you’ve ever found yourself looking up the same question, concept, or syntax over and over again when programming,..."
2,William Koehrsen,2.8K,11,https://towardsdatascience.com/automated-feature-engineering-in-python-99baf11cc219?source=---------2----------------,Automated Feature Engineering in Python – Towards Data Science,Machine learning is increasingly moving from hand-designed models to automatically optimized pipelines using tools s...
3,Gant Laborde,1.3K,7,https://medium.freecodecamp.org/machine-learning-how-to-go-from-zero-to-hero-40e26f8aa6da?source=---------3---------...,Machine Learning: how to go from Zero to Hero – freeCodeCamp,"If your understanding of A.I. and Machine Learning is a big question mark, then this is the blog post for you. Here,..."
4,Emmanuel Ameisen,935,11,https://blog.insightdatascience.com/reinforcement-learning-from-scratch-819b65f074d8?source=---------4----------------,Reinforcement Learning from scratch – Insight Data,"Want to learn about applied Artificial Intelligence from leading practitioners in Silicon Valley, New York, or Toron..."


---
## 4. StackOverflow Programming Questions 2020-2025
`kutayahin/stackoverflow-programming-questions-2020-2025`

Preguntas recientes de Stack Overflow (2020-2025). Útil para tener contenido técnico actualizado y evitar sesgo hacia tecnologías antiguas del StackSample original.

In [9]:
# Set the path to the file you'd like to load
file_path = "stackoverflow_combined.csv"

# Leemos directo del archivo ya descargado por la celda 0.1
# (el dataset también trae stackoverflow_combined_info.json y
# stackoverflow_combined_statistics.json con metadatos, no se usan aquí)
df_so_recent = pd.read_csv(
    os.path.join(rutas_locales["kutayahin/stackoverflow-programming-questions-2020-2025"], file_path)
)

resumenes.append(perfilar("kutayahin/stackoverflow-programming-questions-2020-2025", df_so_recent))

=== kutayahin/stackoverflow-programming-questions-2020-2025 ===
Shape: (95636, 34)

Columnas y tipos:
question_id                      int64
title                           object
body                            object
tags                            object
tag_count                        int64
programming_language            object
categories                      object
creation_date                   object
creation_year                    int64
creation_month                   int64
creation_weekday                 int64
last_activity_date              object
view_count                       int64
score                            int64
answer_count                     int64
comment_count                    int64
favorite_count                   int64
is_answered                       bool
has_accepted_answer               bool
accepted_answer_score            int64
has_code                          bool
code_block_count                 int64
title_word_count                 int64
t

,question_id,title,body,tags,tag_count,programming_language,categories,creation_date,creation_year,creation_month,creation_weekday,last_activity_date,view_count,score,answer_count,comment_count,favorite_count,is_answered,has_accepted_answer,accepted_answer_score,has_code,code_block_count,title_word_count,title_char_count,body_word_count,body_char_count,difficulty_score,quality_score,owner_reputation,owner_badge_count,first_response_time_seconds,first_response_time_hours,top_answer_score,top_answer_body_length
0,79810291,AttributeError: 'NoneType' object has no attribute 'get' in request from AI Tools,I am trying to develop a tool that agent (CodeAgent) will call on user input (prompt) in Gradio. This tool will inte...,"python, artificial-intelligence, huggingface, gradio",4,python,"bug, api, backend",2025-11-05 15:49:00,2025,11,2,2025-11-05 15:57:09,18,-1,0,0,0,False,False,0,True,2,12,81,36,190,0.200,0.70,345,0,NaN,NaN,0,0
1,79810195,"contextlib tries to change a ""frozen"" exception",I got a strange error in my pytest after some Dependabot updates. In my case it's triggered by an non-existent MinIO...,"python, minio",2,python,"bug, testing",2025-11-05 14:23:06,2025,11,2,2025-11-05 14:25:06,29,0,0,0,0,False,False,0,True,3,7,47,113,599,0.200,0.70,529,0,NaN,NaN,0,0
2,79810063,Tkinter window shrinks after embedding Matplotlib canvas — how to prevent it?,I'm building a multi-frame Tkinter app for a school project (NEA dashboard). One of the frames displays a Matplotlib...,"python, matplotlib, tkinter",3,python,"api, backend",2025-11-05 12:42:01,2025,11,2,2025-11-05 12:42:01,27,2,0,0,0,False,False,0,True,11,12,77,143,812,0.201,0.70,21,0,NaN,NaN,0,0
3,79810057,"CatalystAppError: {'code': 'FATAL ERROR', 'message': 'Catalyst headers are empty'} when initializing Catalyst app",I’m working on a chatbot project using the Zoho Catalyst SDK. My goal is to use the Catalyst Cache service to store ...,"python, zoho, zohocatalyst",3,python,bug,2025-11-05 12:36:07,2025,11,2,2025-11-05 12:45:43,48,0,0,0,0,False,False,0,True,3,13,113,124,905,0.202,0.68,9,0,NaN,NaN,0,0
4,79810049,Sympy : Problem with simplifying a Sympy vector from user-defined transformation,"I am creating my own coordinate system using the vector module, and came across some strange behavior. First, the fo...","python, sympy",2,python,bug,2025-11-05 12:28:43,2025,11,2,2025-11-05 14:38:19,44,1,0,0,0,False,False,0,True,7,11,80,109,710,0.199,0.70,1620,0,NaN,NaN,0,0


---
## 5. Text Document Classification Dataset
`sunilthite/text-document-classification-dataset`

Dataset genérico de clasificación de documentos por categoría. Sirve como referencia/baseline para el pipeline de clasificación, aunque no sea 100% contenido técnico.

In [10]:
# Set the path to the file you'd like to load
file_path = "df_file.csv"

# Leemos directo del archivo ya descargado por la celda 0.1
df_text_classification = pd.read_csv(
    os.path.join(rutas_locales["sunilthite/text-document-classification-dataset"], file_path)
)

resumenes.append(perfilar("sunilthite/text-document-classification-dataset", df_text_classification))

=== sunilthite/text-document-classification-dataset ===
Shape: (2225, 2)

Columnas y tipos:
Text     object
Label     int64
dtype: object

Nulos por columna:
Text     0
Label    0
dtype: int64

Primeros 5 registros:


,Text,Label
0,Budget to set scene for election\n \n Gordon Brown will seek to put the economy at the centre of Labour's bid for a ...,0
1,Army chiefs in regiments decision\n \n Military chiefs are expected to meet to make a final decision on the future o...,0
2,Howard denies split over ID cards\n \n Michael Howard has denied his shadow cabinet was split over its decision to b...,0
3,Observers to monitor UK election\n \n Ministers will invite international observers to check the forthcoming UK gene...,0
4,Kilroy names election seat target\n \n Ex-chat show host Robert Kilroy-Silk is to contest the Derbyshire seat of Ere...,0


---
## 6. Comparativa y decisión

Tabla resumen de dimensiones y columnas de todos los datasets cargados, para decidir cuál(es) usar en el pipeline de `TechMind AI`.

In [11]:
df_resumen = pd.DataFrame(resumenes)
df_resumen

,dataset,filas,columnas,nombres_columnas
0,fabiochiusano/medium-articles,192368,6,"[title, text, url, authors, timestamp, tags]"
1,hsankesara/medium-articles,337,6,"[author, claps, reading_time, link, title, text]"
2,kutayahin/stackoverflow-programming-questions-2020-2025,95636,34,"[question_id, title, body, tags, tag_count, programming_language, categories, creation_date, creation_year, creation..."
3,sunilthite/text-document-classification-dataset,2225,2,"[Text, Label]"


### Criterios sugeridos para elegir el/los dataset(s) final(es)

- **Relevancia temática**: ¿el contenido es técnico (código, documentación, tutoriales) o genérico?
- **Calidad de las etiquetas**: ¿existen categorías/tags claros y consistentes para usar como target?
- **Volumen y balance de clases**: suficientes ejemplos por categoría, sin desbalance extremo.
- **Actualidad**: contenido reciente evita sesgo hacia tecnologías obsoletas.
- **Longitud y limpieza del texto**: impacta directamente en el preprocesamiento (Etapa 3).

Una vez decidido, guardar el subconjunto final (crudo) en `dataset/raw/` y, tras la limpieza (Etapa 3), el resultado en `dataset/processed/`.